In [7]:
import json
import random
from IPython.display import display, HTML

with open("all_arguments.json", "r", encoding="utf-8") as f:
    summary_data = json.load(f)
with open("paragraphs.json", "r", encoding="utf-8") as f:
    source_data = json.load(f)

key_field = "page"  
source_lookup = {entry[key_field]: entry["text"] for entry in source_data if key_field in entry}
sample = random.sample(summary_data, min(len(summary_data) ,30))

def show_scrollable(text, label="Text"):
    display(HTML(f"<b>{label}:</b><div style='max-height:200px; overflow:auto; border:1px solid #ccc; padding:10px'>{text}</div>"))

print("\n=== MANUAL VALIDATION START ===")
relevant_count = 0
missing_count = 0

for i, entry in enumerate(sample, 1):
    source_key = entry.get(key_field)
    original_text = source_lookup.get(source_key)
    print(f"\n--- SAMPLE {i} ---")
    if original_text:
        show_scrollable(original_text, "Original")
    else:
        print(f"[MISSING TEXT for key: {source_key}]")
        missing_count += 1
        continue

    show_scrollable(entry["summary"], "Summary")
    answer = input("Is this summary relevant to the excerpt? (y/n): ").strip().lower()
    if answer == 'y':
        relevant_count += 1

relevance_at_10 = (relevant_count / (len(sample) - missing_count)) * 100 if (len(sample) - missing_count) > 0 else 0
print("Correct: " , relevant_count  , " |  Incorrect :" , len(sample)-relevant_count)
print(f"\n Relevance Score: {relevance_at_10:.2f}% based on {len(sample) - missing_count} samples.")




=== MANUAL VALIDATION START ===

--- SAMPLE 1 ---



--- SAMPLE 2 ---



--- SAMPLE 3 ---



--- SAMPLE 4 ---



--- SAMPLE 5 ---


...


--- SAMPLE 21 ---



--- SAMPLE 22 ---



--- SAMPLE 23 ---



--- SAMPLE 24 ---


Correct:  22  |  Incorrect : 2

 Relevance Score: 91.67% based on 24 samples.


In [ ]:
import json
import re
from fpdf import FPDF
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ====== 1. Citation Matching Check ======
with open("all_arguments.json", "r", encoding="utf-8") as f:
    all_arguments = json.load(f)

with open("line.json", "r", encoding="utf-8") as f:
    lines = json.load(f)

ocr_lookup = {}
for entry in lines:
    page_no = entry["page_no"]
    text = entry["text"].strip()
    ocr_lookup.setdefault(page_no, set()).add(text)

def extract_start_end_page(page_str):
    match_full = re.match(r"(\[p\d+ \d+\])-(\[p\d+ \d+\])", page_str)
    if match_full:
        return match_full.group(1), match_full.group(2)
    match_range = re.match(r"(\[p\d+ )(\d+)-(\d+)\]", page_str)
    if match_range:
        start = f"{match_range.group(1)}{match_range.group(2)}]"
        end = f"{match_range.group(1)}{match_range.group(3)}]"
        return start, end
    return None, None

total = 0
correct = 0
mismatch_pages = []

for arg in all_arguments:
    total += 1
    start_page_no, end_page_no = extract_start_end_page(arg.get("page", "").strip())
    citation_start = arg.get("citation_start", "").strip()
    citation_end = arg.get("citation_end", "").strip()

    start_match = start_page_no in ocr_lookup and citation_start in ocr_lookup[start_page_no]
    end_match = end_page_no in ocr_lookup and citation_end in ocr_lookup[end_page_no]

    if start_match and end_match:
        correct += 1
    else:
        mismatch_pages.append(arg.get("page", ""))

correctness_percentage = (correct / total * 100) if total > 0 else 0

sample = len(sample)
relevance_at_10 = (relevant_count / (sample - missing_count)) * 100 if (sample - missing_count) > 0 else 0

# ====== 3. Create Combined PDF Report ======
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# Title
pdf.set_font("Arial", "B", 16)
pdf.cell(0, 10, "Validation Report", ln=True)

# Citation Match Section
pdf.set_font("Arial", "B", 14)
pdf.cell(0, 10, "1. Citation Match Results", ln=True)
pdf.set_font("Arial", "", 12)
pdf.cell(0, 10, f"Total arguments: {total}", ln=True)
pdf.cell(0, 10, f"Correct matches: {correct}", ln=True)
pdf.cell(0, 10, f"Incorrect matches: {len(mismatch_pages)}", ln=True)
pdf.cell(0, 10, f"Correctness: {correctness_percentage:.2f}%", ln=True)

pdf.ln(6)
pdf.set_font("Arial", "B", 12)
pdf.cell(0, 10, "Mismatched Page Numbers:", ln=True)
pdf.set_font("Arial", "", 10)
if mismatch_pages:
    for page_str in mismatch_pages:
        pdf.cell(0, 6, page_str, ln=True)
else:
    pdf.cell(0, 6, "None", ln=True)

# Relevance Validation Section
pdf.ln(10)
pdf.set_font("Arial", "B", 14)
pdf.cell(0, 10, "2. Manual Relevance Validation Results", ln=True)
pdf.set_font("Arial", "", 12)
pdf.cell(0, 10, f"Sample size: {sample}", ln=True)
pdf.cell(0, 10, f"Correct (Relevant): {relevant_count}", ln=True)
pdf.cell(0, 10, f"Incorrect (Not Relevant): {sample - relevant_count - missing_count}", ln=True)
pdf.cell(0, 10, f"Missing: {missing_count}", ln=True)
pdf.cell(0, 10, f"Relevance Score: {relevance_at_10:.2f}%", ln=True)

# Save PDF
pdf.output("validation_report.pdf")
print("Combined PDF saved as validation_report.pdf")


Combined PDF saved as validation_report.pdf
